## Step I. Environment and Data Setup

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import warnings
warnings.filterwarnings("ignore")

The project uses the Brazilian E-Commerce Public Dataset by Olist, which contains information about orders, customers, products, sellers, payments, and customer reviews.

The datasets were loaded separately and later combined into a analytical master table for business analysis and dashboard creation.

In [2]:
# Loading raw datasets 
orders       = pd.read_csv("data/olist_orders_dataset.csv")
customers    = pd.read_csv("data/olist_customers_dataset.csv")
order_items  = pd.read_csv("data/olist_order_items_dataset.csv")
products     = pd.read_csv("data/olist_products_dataset.csv")
sellers      = pd.read_csv("data/olist_sellers_dataset.csv")
payments     = pd.read_csv("data/olist_order_payments_dataset.csv")
reviews      = pd.read_csv("data/olist_order_reviews_dataset.csv")
categories   = pd.read_csv("data/product_category_name_translation.csv")

## Step II. Exploratory Data Analysis

Before cleaning and transforming the datasets, an initial exploratory analysis was performed to better understand the structure and quality of the raw data.

For each dataset, the following checks were conducted:

- dataset dimensions
- column data types
- missing values
- duplicate records
- sample observations

This step helps identify potential data quality issues and guides subsequent cleaning and feature engineering processes.

In [3]:
# Exploratory Data Analysis 
def explore_df(df, name):
    """Print shape, dtypes, sample, null counts for any dataframe."""
    print(f"\n{'='*50}")
    print(f"DATAFRAME NAME: {name}")
    print(f"{'='*50}")
    print(f"\nSHAPE: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\nCOLUMN TYPES:\n{df.dtypes}")
    nulls = df.isnull().sum()
    print(f"\nMISSING VALUES:\n{nulls[nulls > 0] if nulls.any() else 'None'}")
    print(f"\nDUPLICATES: {df.duplicated().sum()}")
    display(df.head(3))

The exploration was applied consistently across all source tables, including orders, customers, products, sellers, payments, and customer reviews.

In [4]:
for df, name in [
    (orders,      "orders"),
    (customers,   "customers"),
    (order_items, "order_items"),
    (products,    "products"),
    (sellers,     "sellers"),
    (payments,    "payments"),
    (reviews,     "reviews"),
    (categories,  "categories"),
]:
    explore_df(df, name)


DATAFRAME NAME: orders

SHAPE: 99,441 rows × 8 columns

COLUMN TYPES:
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object

MISSING VALUES:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

DUPLICATES: 0


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00



DATAFRAME NAME: customers

SHAPE: 99,441 rows × 5 columns

COLUMN TYPES:
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

MISSING VALUES:
None

DUPLICATES: 0


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP



DATAFRAME NAME: order_items

SHAPE: 112,650 rows × 7 columns

COLUMN TYPES:
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

MISSING VALUES:
None

DUPLICATES: 0


...

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71



DATAFRAME NAME: reviews

SHAPE: 99,224 rows × 7 columns

COLUMN TYPES:
review_id                  object
order_id                   object
review_score                int64
review_comment_title       object
review_comment_message     object
review_creation_date       object
review_answer_timestamp    object
dtype: object

MISSING VALUES:
review_comment_title      87656
review_comment_message    58247
dtype: int64

DUPLICATES: 0


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24



DATAFRAME NAME: categories

SHAPE: 71 rows × 2 columns

COLUMN TYPES:
product_category_name            object
product_category_name_english    object
dtype: object

MISSING VALUES:
None

DUPLICATES: 0


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


## Step III. Data Cleaning and Preparation

After completing the exploratory data analysis (EDA), the next step was to prepare the datasets for analytical work.  
The objective of data cleaning was not only to remove inconsistencies but also to preserve meaningful business information.

### 1. Data Type Conversion
Several columns containing timestamps were initially stored as string values. These columns were converted to datetime format to enable time-based analysis.

Converted columns:

- order_purchase_timestamp
- order_approved_at
- order_delivered_carrier_date
- order_delivered_customer_date
- order_estimated_delivery_date
- review_creation_date
- shipping_limit_date
- review_creation_date
- review_answer_timestamp

In [5]:
# Changing date an time columns type to datetime format
def parse_date_columns(df, date_cols):
    """Convert specified columns to datetime format."""
    df[date_cols] = df[date_cols].apply(pd.to_datetime, errors="coerce")
    return df


# DataFrame: orders
orders = parse_date_columns(orders, [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
])

# DataFrame: order_items
order_items = parse_date_columns(order_items, ["shipping_limit_date"])

# DataFrame: reviews
reviews = parse_date_columns(reviews, ["review_creation_date", "review_answer_timestamp"])

# Checking the result
date_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

date_cols_review = ["review_creation_date", "review_answer_timestamp"]

print(f"{orders[date_cols_orders].dtypes}")
print(f"\nshipping_limit_date    {order_items["shipping_limit_date"].dtypes}")
print(f"\n{reviews[date_cols_review].dtypes}")


order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

shipping_limit_date    datetime64[ns]

review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object


### 2. Handling Missing Values
Missing values needed to be handled individually instead of being removed automatically.

**Orders dataset**

Review of missing delivery timestamps showed that most records correspond to orders with statuses such as "shipped", "processing", "canceled", or "unavailable". Therefore, missing delivery dates were treated as valid business scenarios rather than data errors.

A small number of inconsistent records were identified where orders were marked as "delivered" despite missing customer delivery timestamps.

The overview of the records with status "delivered" and missing values showed that most likely customers had not marked the order as received on their side, or that might have been a system bug.

Since those records represented less than 0.01% of the dataset and could negatively affect delivery time calculations, they were removed from the analysis by creating a delivery flag.

In [6]:
# Checking statuses of the missing values
orders[orders["order_delivered_customer_date"].isna()]["order_status"].value_counts()

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

In [7]:
# Viewing records in status "delivered" but having missing values in "order_delivered_customer_date" column
orders[
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].isna())
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaT,NaT,2017-06-23
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19


In [8]:
# Number of orders in status "delivered"
orders[orders["order_status"] == "delivered"]["order_status"].value_counts()

order_status
delivered    96478
Name: count, dtype: int64

In [9]:
# Delivery flag
orders["is_delivered"] = (
    (orders["order_status"] == "delivered") &
    (orders["order_delivered_customer_date"].notna())
)
orders["is_delivered"].value_counts()

is_delivered
True     96470
False     2971
Name: count, dtype: int64

In [10]:
# For our analysis goals we need only orders in "delivered" status, so creating new dataframe
orders_delivered = orders[orders["is_delivered"]].copy()

print(f"Orders total:    {len(orders):,}")
print(f"Orders delivered: {len(orders_delivered):,}")
print(f"Removed:         {len(orders) - len(orders_delivered):,}")

Orders total:    99,441
Orders delivered: 96,470
Removed:         2,971


**Products dataset**

Analysis showed that products with missing metadata were still actively used in customer transactions and generated revenue.

A total of 1,603 order items were linked to products with missing category and description information. Therefore, removing these rows would lead to revenue loss and biased business analysis, so the best option was to replace missing product categories  with 'unknown' value and missing numerical data with median values.


In [11]:
# Overviewing products w/o category 
products[products["product_category_name"].isnull()].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0


In [12]:
# Checking if these products are getting ordered
missing_products = products[
    products["product_category_name"].isnull()
]

order_items[
    order_items["product_id"].isin(
        missing_products["product_id"]
    )
].shape

(1603, 7)

As we can see 1603 orders were made for the products with NaN values. So, we need to preserve these orders for accurate analysis. 

In [13]:
# Filling missing values with "unknown"
products["product_category_name"] = (
    products["product_category_name"]
    .fillna("unknown")
)

# Replacing numerical characteristics with the median
numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]


for col in numeric_cols:
    products[col] = products[col].fillna(products[col].median())

print(products.isnull().sum())

product_id                    0
product_category_name         0
product_name_lenght           0
product_description_lenght    0
product_photos_qty            0
product_weight_g              0
product_length_cm             0
product_height_cm             0
product_width_cm              0
dtype: int64


**Reviews dataset**

Out of 99,224 reviews, only 11.7% contain a title and 41.3% contain a message.
This is an expected behavior for most of the customers. They simply leave a star rating without written feedback.

Since`review_comment_title` and `review_comment_message` columns will not be used here for further analysis there is no need of further cleaning manipulations with this data. Only `review_score` will be used.


In [14]:
# How many reviews in total have some text?
total = len(reviews)
has_title = reviews["review_comment_title"].notna().sum()
has_message = reviews["review_comment_message"].notna().sum()

print(f"Total reviews:      {total:,}")
print(f"With title:         {has_title:,} ({has_title/total*100:.1f}%)")
print(f"With message:       {has_message:,} ({has_message/total*100:.1f}%)")

Total reviews:      99,224
With title:         11,568 (11.7%)
With message:       40,977 (41.3%)


### 3. Feature Engineering

To support further business analysis, several new analytical features were created across the datasets.

**Orders Dataset:** 

The following delivery and time-related features were added:

- `delivery_delay_days` — number of days between purchase and actual delivery
- `is_late` — boolean flag: True if delivered after the estimated delivery date
- `purchase_year_month` — year and month of purchase for time series analysis
- `purchase_month` / `purchase_year` — extracted from the purchase timestamp to analyze seasonality and yearly trends

These features support logistics performance analysis, customer satisfaction evaluation, and revenue trend exploration.

**Payments Dataset:**

Payment information was aggregated to the order level:

- `payments_agg` — payments aggregated to one row per order (total value)

This step was necessary because some orders contain multiple payment transactions.

**Reviews Dataset:**

A sentiment grouping feature was created based on review scores:

- `review_sentiment_group`
  - Positive → review scores 4–5
  - Neutral → review score 3
  - Negative → review scores 1–2

This feature simplifies customer satisfaction analysis and allows easier segmentation of review behavior.

In [15]:
# Features for revenue over time and seasonality analysis
orders_delivered["purchase_year_month"] = (
    orders_delivered["order_purchase_timestamp"].dt.to_period("M").astype(str)
)
orders_delivered["purchase_month"] = orders_delivered["order_purchase_timestamp"].dt.month
orders_delivered["purchase_year"] = orders_delivered["order_purchase_timestamp"].dt.year


In [16]:
# Delivery delay vs review score
orders_delivered["delivery_delay_days"] = (
    orders_delivered["order_delivered_customer_date"]
    - orders_delivered["order_purchase_timestamp"]
).dt.days

orders_delivered["is_late"] = (
    orders_delivered["order_delivered_customer_date"].dt.date
    > orders_delivered["order_estimated_delivery_date"].dt.date
)

In [17]:
orders_delivered.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,purchase_year_month,purchase_month,purchase_year,delivery_delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,True,2017-10,10,2017,8,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,True,2018-07,7,2018,13,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,True,2018-08,8,2018,9,False


In [18]:
# Payments aggregation
payments_agg = (
    payments.groupby("order_id")
    .agg(
        total_order_value=("payment_value", "sum"),
        payment_type=("payment_type", "first"),
        installments=("payment_installments", "max")
    )
    .reset_index()
)

payments_agg.head(3)

,order_id,total_order_value,payment_type,installments
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,credit_card,2
1,00018f77f2f0320c557190d7a144bdd3,259.83,credit_card,3
2,000229ec398224ef6ca0657da4fc703e,216.87,credit_card,5


In [19]:
# Review sentiment 
reviews["review_sentiment_group"] = np.where(
    reviews["review_score"] >= 4, "Positive",
    np.where(reviews["review_score"] == 3, "Neutral", "Negative")
)

reviews.head(3)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_sentiment_group
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59,Positive
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13,Positive
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24,Positive


## Step IV. Creating Master Table

A master table `master_df` was created by joining all relevant tables using `order_items` as the foundation because it has the most detailed data — one row for every item in an order.

**Joins applied:**
- `orders_delivered` — order-level info: dates, delivery delay, late flag
- `payments_agg` — total order value aggregated from payments
- `customers` — customer state
- `products_en` — product category in English
- `reviews` — review score and sentiment group (left join — not all orders have a review)
- `sellers` — seller state

**Result:** 110,186 rows × 22 columns

**Remaining nulls after cleaning:**
- `review_score` / `review_sentiment_group` — 827 nulls, expected as not all orders have a review
- `product_category_name_english` — filled with `"unknown"` for products without a category

In [20]:
# Merging products with English categories
products_en = products.merge(categories, on="product_category_name", how="left")
print(f"products_en: {len(products_en):,}")

products_en: 32,951


In [21]:
products_en.head(3)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure


In [22]:
# Merging tabels into a Master table
master_df = (
    order_items
    .merge(orders_delivered[["order_id", "customer_id", 
                              "order_purchase_timestamp",
                              "purchase_year_month", "purchase_month", 
                              "purchase_year", "delivery_delay_days", 
                              "is_late"]], on="order_id")
    .merge(payments_agg, on="order_id")
    .merge(customers[["customer_id", "customer_state"]], on="customer_id")
    .merge(products_en[["product_id", "product_category_name_english"]], on="product_id")
    .merge(reviews[["order_id", "review_score", "review_sentiment_group"]]
           .drop_duplicates("order_id"), on="order_id", how="left")
    .merge(sellers[["seller_id", "seller_state"]], on="seller_id")
)

print(f"Master table shape: {master_df.shape}")
master_df.head()

Master table shape: (110186, 22)


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_purchase_timestamp,purchase_year_month,...,delivery_delay_days,is_late,total_order_value,payment_type,installments,customer_state,product_category_name_english,review_score,review_sentiment_group,seller_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,2017-09,...,7,False,72.19,credit_card,2,RJ,cool_stuff,5.0,Positive,SP
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,2017-04,...,16,False,259.83,credit_card,3,SP,pet_shop,4.0,Positive,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,2018-01,...,7,False,216.87,credit_card,5,MG,furniture_decor,5.0,Positive,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35,2018-08,...,6,False,25.78,credit_card,2,SP,perfumery,4.0,Positive,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,2017-02,...,25,False,218.04,credit_card,3,SP,garden_tools,5.0,Positive,PR


In [23]:
master_df.isnull().sum()[master_df.isnull().sum() > 0]

product_category_name_english    1559
review_score                      827
review_sentiment_group            827
dtype: int64

In [24]:
# Checking which product_ids don't have a category in English
master_df[master_df["product_category_name_english"].isnull()][["product_id", "product_category_name_english"]].head()

,product_id,product_category_name_english
121,ff6caf9340512b8bf6d2a2a6df032cfa,NaN
123,a9c404971d1a5b1cbc2e4070e02731fd,NaN
130,5a848e4ab52fd5445cdc07aab1c40e48,NaN
140,41eee23c25f7a574dfaf8d5c151dbb12,NaN
169,e10758160da97891c2fdcbc35f0f031d,NaN


In [25]:
# Checking original category name for missing values
master_df[master_df["product_category_name_english"].isnull()][["product_id"]].merge(
    products_en[["product_id", "product_category_name", "product_category_name_english"]], 
    on="product_id"
).head(10)

,product_id,product_category_name,product_category_name_english
0,ff6caf9340512b8bf6d2a2a6df032cfa,unknown,NaN
1,a9c404971d1a5b1cbc2e4070e02731fd,unknown,NaN
2,5a848e4ab52fd5445cdc07aab1c40e48,unknown,NaN
3,41eee23c25f7a574dfaf8d5c151dbb12,unknown,NaN
4,e10758160da97891c2fdcbc35f0f031d,unknown,NaN
5,76d1a1a9d21ab677a61c3ae34b1b352f,unknown,NaN
6,fbb1cfc2810efabf3235eccf4530f4ae,unknown,NaN
7,5a848e4ab52fd5445cdc07aab1c40e48,unknown,NaN
8,9f69acd4da62618a3f6365b732d00ccd,unknown,NaN
9,ea11e700a343582ad56e4c70e966cb36,unknown,NaN


In [26]:
# Filling NaN values in product_category_name_english column with 'unknown' values
master_df["product_category_name_english"] = (
    master_df["product_category_name_english"].fillna("unknown")
)

master_df.isnull().sum()[master_df.isnull().sum() > 0]

review_score              827
review_sentiment_group    827
dtype: int64

**`repeat_customers` table**

A separate aggregation was created to analyze repeat purchase behavior. `repeat_customers` was built from `orders_delivered` at the customer level — one row per customer with the total number of unique orders.

A customer is considered a repeat buyer if they placed more than one order (`is_repeat_customer = True`).

This table was intentionally kept separate from `master_df` as it operates at the customer level, while the master table is at the order-item level.

In [27]:
# DataFrame for repeat purchase behavior analysis
repeat_customers = (
    orders_delivered
    .merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
    .groupby("customer_unique_id")["order_id"]
    .nunique()
    .reset_index()
    .rename(columns={"order_id": "order_count"})
)
repeat_customers["is_repeat_customer"] = repeat_customers["order_count"] > 1

# Checking if it was created correctly
repeat_customers.describe()


,order_count
count,93350.000000
mean,1.033423
std,0.209106
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,15.000000


In [28]:
# Data quality check — orders without payment record
order_id = "bfbd0f9bdef84302105ad712db648a6c"

orders_without_payment = orders_delivered[
    ~orders_delivered["order_id"].isin(payments["order_id"])
]
print(f"Orders without payment: {len(orders_without_payment)}")

Orders without payment: 1


**Note:** 1 delivered order was excluded from the Master Table due to missing payment record. This is a data quality issue in the source dataset and has no material impact on the analysis.

Before performing the analysis, incomplete historical periods were identified and excluded to ensure consistency and reliability of business metrics.

Excluded periods:
- `2016-10` — partial first month in dataset (265 orders)
- `2016-12` — only 1 recorded order (likely missing historical data)

All subsequent analyses are based on the validated period from January 2017 to August 2018.

In [29]:
# Filter valid analysis period
clean_df = master_df[
    (master_df["order_purchase_timestamp"] >= "2017-01-01") &
    (master_df["order_purchase_timestamp"] <= "2018-08-31")
].copy()

print("Analysis period:")
print(
    clean_df["order_purchase_timestamp"].min(),
    "→",
    clean_df["order_purchase_timestamp"].max()
)

print(f"\nMaster tabel dataset shape: {master_df.shape}")
print(f"\nFiltered dataset shape: {clean_df.shape}")

Analysis period:
2017-01-05 11:56:06 → 2018-08-29 15:00:37

Master tabel dataset shape: (110186, 22)

Filtered dataset shape: (109872, 22)


## Exporting prepared datasets for further analysis 

In [31]:
# clean_df.to_csv("clean_df.csv", index=False)
# repeat_customers.to_csv("repeat_customers.csv", index=False)